# Riset Model_SmartSplit Bill

Notebook ini digunakan untuk membandingkan dua model untuk data struk yang bersifat OCR-free, yaitu **Donut** (model lokal) dan **Gemini** (model API). Pengujian dilakukan pada dua foto struk yang berbeda. Aspek yang dibandingkan adalah kualitas hasil pembacaan dan kecepatan. Hasil perbandingan menjadi dasar pemilihan satu model untuk digunakan pada prototype aplikasi.

## 1. Persiapan Library

In [10]:
!pip install -q langchain-google-genai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00


In [ ]:
import time
from PIL import Image

import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

## 2. Load Foto Struk

In [ ]:
bill1_path = "bill1.jpg"
bill2_path = "bill2.jpg"

bill1 = Image.open(bill1_path).convert("RGB")
bill2 = Image.open(bill2_path).convert("RGB")

print("struk 1:", bill1.size)
print("struk 2:", bill2.size)

struk 1: (362, 450)
struk 2: (800, 800)


## 3. Model 1_Donut (lokal)

In [ ]:
donut_model_name = "naver-clova-ix/donut-base-finetuned-cord-v2"
donut_processor = DonutProcessor.from_pretrained(donut_model_name)
donut_model = VisionEncoderDecoderModel.from_pretrained(donut_model_name)
print("donut siap")

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

donut siap


In [ ]:
def extract_donut(image):
    task_prompt = "<s_cord-v2>"
    decoder_input_ids = donut_processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids
    pixel_values = donut_processor(image, return_tensors="pt").pixel_values

    start = time.time()
    outputs = donut_model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=donut_model.decoder.config.max_position_embeddings,
        pad_token_id=donut_processor.tokenizer.pad_token_id,
        eos_token_id=donut_processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[donut_processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    elapsed = time.time() - start

    sequence = donut_processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(donut_processor.tokenizer.eos_token, "")
    sequence = sequence.replace(donut_processor.tokenizer.pad_token, "")

    result = donut_processor.token2json(sequence)
    return result, elapsed

In [ ]:
# jalankan donut ke struk 1
donut_result1, donut_time1 = extract_donut(bill1)
print("waktu inferensi:", round(donut_time1, 2), "detik")
donut_result1

waktu inferensi: 23.0 detik


{'menu': [{'nm': 'Uluwatu',
   'unitprice': 'Cafe',
   'cnt': 'Drifter Cafe',
   'price': 'Uluwatu'},
  {'nm': 'Denpasar, Bali',
   'unitprice': '80361',
   'cnt': 'Pented 23',
   'price': '0817557111'},
  {'nm': 'August 2019 20.59',
   'unitprice': '20:59',
   'cnt': '23',
   'price': '107808'},
  {'nm': 'Table: 27, 1 guest',
   'cnt': '23',
   'price': {'unitprice': '10R98.000,00', 'cnt': '3'}},
  {'nm': 'Black Pepper Crusted',
   'unitprice': '104.000,00',
   'cnt': '3',
   'price': '104.000,00'},
  {'nm': 'Jackfruit Tacos',
   'unitprice': '88.000,00',
   'cnt': '3',
   'price': '88.000,00'},
  {'nm': 'Lemon cheesecake',
   'unitprice': '10R40.000,00',
   'cnt': '3',
   'price': '88.000,00'}],
 'sub_total': {'subtotal_price': '19R290.0000,00'},
 'total': {'total_price': '230.000,00',
  'cashprice': 'IDR35.640,00',
  'changeprice': '400,00',
  'creditcardprice': '392.040,00'}}

In [ ]:
# jalankan donut ke struk 2
donut_result2, donut_time2 = extract_donut(bill2)
print("waktu inferensi:", round(donut_time2, 2), "detik")
donut_result2

waktu inferensi: 30.13 detik


{'menu': [{'nm': 'Jl. Medayu Utara',
   'unitprice': 'Shop',
   'cnt': '3',
   'price': 'Surat -'},
  {'nm': '8152962022041414 -', 'num': '2022-04-14', 'price': 'Afi'},
  {'nm': '14: 24: 34', 'num': 'sheila', 'price': '202-24'},
  {'nm': 'Nasi Ayam Geprek', 'price': 'Rp 12.000'},
  {'nm': 'Nasi Ayam Kremes',
   'unitprice': '15.000',
   'cnt': '1 X',
   'price': 'Rp 15.000'},
  {'nm': 'Nasi Goreng Special', 'price': 'Rp 20.000'}],
 'sub_total': {'subtotal_price': '47.000'},
 'total': {'total_price': '47.000', 'cashprice': '47.000', 'changeprice': '0'}}

## 4. Model 2_Gemini (API)

In [ ]:
import os
import json
import base64
from io import BytesIO

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("fazri51")

gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
print("gemini siap")

gemini siap


In [ ]:
# prompt buat nyuruh gemini baca struk terus balikin hasil dalam format json
gemini_prompt = """
Kamu diberikan sebuah gambar struk. Tolong baca isi struk tersebut ke dalam format JSON berikut:

{
    "menus": [
        {"name": <nama_item>, "count": <jumlah_item>, "price": <total_harga_item>}
    ],
    "total": <total_harga_struk>
}

Untuk price/total: jangan gunakan pemisah titik atau koma, cukup angka saja, kecuali untuk angka desimal.
Untuk count: jika jumlah item tidak ditemukan pada struk, anggap nilainya 1.
Tidak perlu memberikan harga satuan.

Kembalikan hasil hanya dalam format JSON.
"""

def extract_gemini(image):
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    message = HumanMessage(
        content=[
            {"type": "text", "text": gemini_prompt},
            {"type": "image_url", "image_url": f"data:image/png;base64,{image_b64}"},
        ]
    )

    start = time.time()
    response = gemini.invoke([message]).content
    elapsed = time.time() - start

    clean = response.replace("```json", "").replace("```", "")
    result = json.loads(clean)
    return result, elapsed

In [ ]:
# jalankan gemini ke struk 1
gemini_result1, gemini_time1 = extract_gemini(bill1)
print("waktu inferensi:", round(gemini_time1, 2), "detik")
gemini_result1

waktu inferensi: 5.37 detik


{'menus': [{'name': 'Battered Mahi Mahi', 'count': 1, 'price': 98000.0},
  {'name': 'Black Pepper Crusted Tuna', 'count': 1, 'price': 104000.0},
  {'name': 'Jackfruit Tacos', 'count': 1, 'price': 88000.0},
  {'name': 'Lemon cheesecake', 'count': 1, 'price': 40000.0}],
 'total': 392040.0}

In [ ]:
# jalankan gemini ke struk 2
gemini_result2, gemini_time2 = extract_gemini(bill2)
print("waktu inferensi:", round(gemini_time2, 2), "detik")
gemini_result2

waktu inferensi: 5.57 detik


{'menus': [{'name': 'Nasi Ayam Geprek', 'count': 1, 'price': 12000},
  {'name': 'Nasi Ayam Kremes', 'count': 1, 'price': 15000},
  {'name': 'Nasi Goreng Spesial', 'count': 1, 'price': 20000}],
 'total': 47000}

## 5. Perbandingan Kecepatan Model

Tabel berikut merangkum kecepatan kedua model pada masing-masing struk.

In [ ]:
print("model  | struk 1 | struk 2")
print("donut  |", round(donut_time1, 2), "  |", round(donut_time2, 2))
print("gemini |", round(gemini_time1, 2), "  |", round(gemini_time2, 2))

model  | struk 1 | struk 2
donut  | 23.0   | 30.13
gemini | 5.37   | 5.57
